# Layer Ablation Effects

What happens when layers are removed? Zero ablation, component ablation,
mean ablation, and cumulative ablation.

## Why This Matters

Ablation layer effects measures the effect of removing or zeroing specific components. Ablation is the simplest causal intervention: if removing a component changes behavior, that component is necessary. Systematic ablation reveals the importance hierarchy of model components.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_ablation_effects import (
    layer_zero_ablation, component_ablation,
    mean_ablation, cumulative_ablation,
    layer_ablation_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Zero Ablation

KL divergence when zeroing each layer's output.

In [ ]:
result = layer_zero_ablation(model, tokens, position=-1)
print(f"Base prediction: {result['base_prediction']}")
for p in result['per_layer']:
    flag = ' [CHANGED]' if p['prediction_changed'] else ''
    bar = '█' * min(int(p['kl_divergence'] * 10), 40)
    print(f"  Layer {p['layer']}: KL={p['kl_divergence']:.4f}{flag} {bar}")

## Component Ablation

Attention vs MLP: which is more important at each layer?

In [ ]:
for layer in range(4):
    result = component_ablation(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: attn_KL={result['attn_kl']:.4f}, "
          f"mlp_KL={result['mlp_kl']:.4f} [{result['more_important'].upper()}]")

## Mean Ablation

Replace with mean instead of zero — less harsh.

In [ ]:
for layer in range(4):
    result = mean_ablation(model, tokens, layer=layer, position=-1)
    flag = ' [CHANGED]' if result['prediction_changed'] else ''
    print(f"  Layer {layer}: KL={result['kl_divergence']:.4f}{flag}")

## Cumulative Ablation

Remove layers from the top — how many can we remove?

In [ ]:
result = cumulative_ablation(model, tokens, position=-1)
for p in result['per_n_layers_removed']:
    flag = ' [CHANGED]' if p['prediction_changed'] else ''
    print(f"  Remove {p['n_layers_removed']} layers: KL={p['kl_divergence']:.4f}{flag}")

## Summary

In [ ]:
result = layer_ablation_summary(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: KL={p['kl_divergence']:.4f}, "
          f"critical={p['more_important']}")